# Capítulo 1 — Modelos de ataque: exemplos adversariais

## Missão
Você entrou no **laboratório de segurança de IA da Fortaleza Digital**. A sua função é atacar os próprios modelos da empresa (o *red team*) antes que alguém de fora o faça, e depois defendê-los. Primeiro alvo: um classificador de dígitos, que faz o papel do leitor de cheques e formulários. Você vai:
- descrever o modelo de ameaça;
- gerar **exemplos adversariais** de caixa branca (FGSM e PGD);
- medir a queda de acurácia em função do orçamento ε;
- atacar de **caixa preta**, por transferência e só por consultas.

**No fim deste caderno você terá:** o módulo `meu_escudo/ataques.py` (gradiente de entrada da regressão softmax, FGSM, PGD e ataque de caixa preta por consultas) e a curva de robustez do modelo.

**Leitura essencial (opcional, livro em inglês):**
- 1.2 Background (attack taxonomy) · PDF p. 18
- 1.3 White-Box Adversarial Attacks · PDF p. 21
- 1.4 Black-Box Adversarial Attacks · PDF p. 31
- 3.2 Categories of Attacks · PDF p. 67
- 3.4 Specific Attacks In The Real World · PDF p. 73

**Como usar:** cada passo tem uma explicação curta e uma célula vazia. Digite o código e rode com `Shift+Enter`. Travou por mais de 10 minutos? O mesmo passo pronto está em `../resolvidos/cap01_modelos_de_ataque_exemplos_adversariais.ipynb`.

## Preparação
Configuração do caderno (já vem pronta). Rode antes de tudo.

In [ ]:
import os, sys, json, time, math, random
sys.path.insert(0, "../comum")                 # dados.py (dados fictícios do projeto)
sys.path.insert(0, "..")                       # raiz: onde fica o seu pacote meu_escudo/
os.makedirs("../meu_escudo", exist_ok=True)
open("../meu_escudo/__init__.py", "a").close()        # meu_escudo/ é o projeto que você constrói ao longo do livro
os.makedirs("../saidas", exist_ok=True)        # arquivos gerados nos exercícios
from dados import *
import warnings; warnings.filterwarnings("ignore")
%load_ext autoreload
%autoreload 2
print("Pronto. Python", sys.version.split()[0], "| seu pacote: meu_escudo")

## Ideia em 30 segundos
**Modelo de ameaça:** o que o atacante quer (errar qualquer coisa ou errar para uma classe-alvo), o que ele sabe (caixa branca ou preta) e o que pode fazer (mexer na entrada ou nos dados de treino).

**FGSM:** dê um passo do tamanho ε na direção do **sinal do gradiente** da perda em relação à **entrada**: `x' = x + ε·sign(∇x L)`. O **PGD** repete passos pequenos e projeta de volta na bola de raio ε.

Mudanças invisíveis a olho podem enganar o modelo.

### 1.1 **Os ataques.** Crie `ataques.py` com:
- `grad_entrada_softmax(W, b, x, y)`: para a regressão logística multinomial (`p = softmax(xWᵀ + b)`), o gradiente da entropia cruzada em relação a `x` é `(p - onehot(y))·W`;
- `fgsm(W, b, X, y, eps)`: devolve `X'` cortado em [0, 1];
- `pgd(W, b, X, y, eps, passo, iters)`: projeta em `[X - eps, X + eps] ∩ [0, 1]` a cada passo;
- `caixa_preta(prever_proba, x, y, eps, consultas, semente)`: uma busca aleatória que só consulta probabilidades. Em cada consulta, inverte o sinal de um bloco de pixels da perturbação e mantém a mudança se a probabilidade da classe certa cair.

_A célula grava `meu_escudo/ataques.py`: a 1ª linha (`%%writefile`) já vem pronta, digite o código embaixo dela._

In [ ]:
%%writefile ../meu_escudo/ataques.py


### 1.2 **O alvo.** Carregue `digitos()`, separe 70/30 estratificado e treine uma `LogisticRegression(max_iter=3000)` (o modelo de caixa branca) e um `MLPClassifier((64,), max_iter=1000)` (o alvo de caixa preta). Mostre a acurácia limpa de cada um.

### 1.3 **FGSM e PGD.** Para ε em 0, 0,05, 0,1, 0,15, 0,2 e 0,3, meça a acurácia da logística nos exemplos adversariais do teste, com FGSM e com PGD (passo 0,01 e 30 iterações). Desenhe as duas curvas: é a **curva de robustez** do modelo.

### 1.4 **Veja com os próprios olhos.** Mostre 5 dígitos originais e as versões PGD com ε = 0,15, lado a lado (`imshow` 8×8), com a previsão do modelo em cima de cada imagem.

### 1.5 **Caixa preta por transferência.** O atacante não conhece o MLP, mas treina o **próprio** modelo substituto (a logística) com dados parecidos. Os exemplos PGD gerados contra o substituto também enganam o MLP? Meça a acurácia do MLP nos exemplos transferidos para ε = 0,1, 0,2 e 0,3.

### 1.6 **Caixa preta só por consultas.** Agora o atacante só pode **perguntar** ao MLP as probabilidades. Ataque 40 imagens do teste com `caixa_preta` (ε = 0,2, 300 consultas cada) e meça a taxa de sucesso (o MLP passa a errar) e o número total de consultas. Por que limitar consultas (*rate limiting*) é uma defesa prática?

## Fechamento

- O que eu construí hoje:
- O que me surpreendeu:
- O que quero tentar sozinho a seguir:
